# Vision: factored VAE
The implementation now lives in `src/world_models/models/vae.py`; loss and optimizer steps live in `src/world_models/training/vae.py`. Install the repo with `uv sync --extra research --extra dev`, then select its kernel.

This notebook starts with fresh weights. Uploaded notebook outputs do not contain trained weights. Your original learning notebook is preserved under `notebooks/archive/`.

In [ ]:
import torch
from world_models.models.vae import VAE
from world_models.training.vae import vae_loss, train_vae_step
torch.manual_seed(0)
vae = VAE()
x = torch.rand(2, 3, 64, 64)  # Shape check only
x_hat, mu, logvar = vae(x)
print(x_hat.shape, mu.shape, logvar.shape)
assert x_hat.shape == x.shape
assert mu.shape == logvar.shape == (2, 32)

## One training step
Create the optimizer once. Rerunning the next cell resets its optimizer state; repeat only the training call when taking more steps. This basic objective is pedagogical, not the fully matched historical training recipe.

In [ ]:
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-4)

In [ ]:
metrics = train_vae_step(vae, x, optimizer)
print(metrics)

## Sampling versus decoding
The encoder deterministically computes distribution parameters. Sampling introduces randomness; the decoder maps a fixed latent deterministically.

In [ ]:
with torch.no_grad():
    mu, logvar = vae.encode(x)
    z1 = vae.reparameterize(mu, logvar)
    z2 = vae.reparameterize(mu, logvar)
    print("Latent difference:", (z1-z2).abs().mean().item())
    assert torch.allclose(vae.decode(z1), vae.decode(z1))

## Next
Open `01_memory.ipynb`. The VAE encodes a frame; memory must summarize a sequence. No trained vision checkpoint is needed for the first synthetic shape exercises.